### Imports and API Endpoint Definitions

We start by importing the required Python libraries for data extraction, XML parsing, in-memory file handling, tabular transformations, and database connectivity.

We also define the dataset identifiers used to retrieve macroeconomic indicators from the ECB and related statistical sources. These series include:

- External net debt
- External net debt as a percentage of GDP
- Gross external debt
- Inflation
- Exchange rates

These constants are declared at the beginning of the notebook to centralize configuration and make the data extraction workflow easier to maintain and update.

In [1]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from io import StringIO
import psycopg

In [2]:
# External net debt in absolute terms
EXTERNAL_NET_DEBT = "BPS/Q.N.AT+BE+HR+CY+EE+FI+FR+DE+GR+IE+IT+LV+LT+LU+MT+NL+PT+SK+SI+ES+I9.W1.S1.S1.LE.NE.FA._T.FNED._Z.EUR._T._X.N.ALL"
# External net debt as a percentage of GDP
EXTERNAL_NET_DEBT_PERCENTAGE = "BPS/Q.N.AT+BE+HR+CY+EE+FI+FR+DE+GR+IE+IT+LV+LT+LU+MT+NL+PT+SK+SI+ES+I9.W1.S1.S1.LE.NE.FA._T.FNED._Z.EUR_R_B1GQ._T._X.N.ALL"
# Gross external debt in absolute terms
GROSS_EXTERNAL_DEBT = "BPS/Q.N.AT+BE+HR+CY+EE+FI+FR+DE+GR+IE+IT+LV+LT+LU+MT+NL+PT+SK+SI+ES+I9.W1.S1.S1.LE.L.FA._T.FGED._Z.EUR._T._X.N.ALL"
# HICP inflation rate for headline and selected components
INFLATION = "ICP/M.DE+FR+ES+IT+NL+BE+AT+PT+FI+IE+GR+SK+SI+EE+LV+LT+LU+CY+MT+U2.N.000000+FOOD00+NRGY00+SERV00+GOODS00+CP0000+ALCBEV+TOBACC+CLTHES+HOUSNG+FURNSH+HEALTH+TRANS+COMMUN+RECREA+EDUCAT+RESTAU+MISCEL.4.ANR"
# Daily exchange rates of selected currencies against the euro
EXCHANGE_RATE = "EXR/D.BGN+CZK+DKK+HUF+PLN+RON+SEK+CHF+NOK+GBP+ISK+USD+JPY+AUD+CAD+NZD.EUR.SP00.A"

### Main Update Function

This function coordinates the end-to-end update of the exchange rate table.

The workflow consists of four steps:
1. Connect to the database.
2. Identify the next missing observation date.
3. Extract and transform new exchange rate data from the API.
4. Load the new records into PostgreSQL.

The function closes the database connection at the end of the process and returns a completion message.

In [3]:
def update_new_data():

    # Open a connection to the PostgreSQL database
    conn = psycopg.connect(
    host=,
    port=,
    dbname=,
    user=,
    password=
    )

    # Create a cursor to execute database queries
    cursor = conn.cursor()

    # Get the next date after the latest stored observation
    next_from_last_date = new_data(conn, cursor)

    # Request new exchange rate data from the external API
    data_exchange_rate_bce = api_extract_data(EXCHANGE_RATE, start_date = next_from_last_date)

    # Transform the API response into an in-memory CSV buffer
    buffer_exchange_rate_data = create_csv_exchange_rate(data_exchange_rate_bce)

    # Load the processed exchange rate data into the database
    load_exchange_rates(buffer_exchange_rate_data, conn, cursor)

    # Close database resources
    cursor.close()
    conn.close()

    return "Process completed successfully"

SyntaxError: expected argument value expression (2783912202.py, line 5)

#### Retrieve Next Available Date

This function queries the `exchange_rates` table to obtain the next date required for the incremental update process.

It calculates the day immediately following the latest date currently stored in the database and returns it as the extraction start date.

In [ ]:
def new_data(conn, cursor):
    #Check the difference between today and the last day stored in the database.
    diff_days = """
        SELECT (NOW() :: DATE - MAX(DATE))
        FROM exchange_rates;
    """
    diff_exec = cursor.execute(diff_days)
    parsed_diff = diff_exec.fetchone()
    diff = int(parsed_diff[0])
    #print(diff)
    
    if diff > 1:
        # Query the next date after the latest stored observation
        query = """
            SELECT (MAX(DATE) + INTERVAL '1 DAY') ::DATE
            FROM exchange_rates;
        """
        
        # Execute the query and fetch the resulting value
        respuesta = cursor.execute(query)
        parsed_response = respuesta.fetchone()
        
        # Convert the date to string format for API usage
        next_from_last_date = str(parsed_response[0])

        return next_from_last_date
    
    else:
        #Updates the last available date in the database to avoid errors.
        print("Exchange rate data has still not been updated in the API.")
        
        query_alt = """
            SELECT MAX(DATE)
            FROM exchange_rates;
        """
        respuesta = cursor.execute(query_alt)
        parsed_response = respuesta.fetchone()
        
        # Convert the date to string format for API usage
        next_from_last_date = str(parsed_response[0])

        return next_from_last_date

#### Extract Data from the ECB API

This function sends an HTTP request to the ECB Data API using the dataset identifier provided as input.

If a `start_date` is specified, the request is restricted to observations from that date onward, enabling incremental extraction. The function returns the raw API response for downstream processing.

In [ ]:
def api_extract_data(code_data, start_date=None):

    # Define the base endpoint for the ECB Data API
    core_url = "https://data-api.ecb.europa.eu/service/data/"
    url = core_url + code_data

    # Restrict the request to observations from the specified start date
    if start_date is not None:
        url += f"?startPeriod={start_date}"

     # Headers to store the content negotiation setting for the format of the output file.
    headers = {
        "Accept": "application/vnd.sdmx.structurespecificdata+xml;version=2.1"
    }

    # Send request
    response = requests.get(url, headers=headers)
    
    # Stop execution if the HTTP response contains an error
    response.raise_for_status()

    return response

#### Transform API Response into CSV Buffer

This function parses the XML response returned by the ECB API and extracts the exchange rate observations for each available currency series.

For every observation, it collects the reference period, the currency code, and the observed value, and writes the result into an in-memory CSV buffer. The output is then returned for subsequent loading into the database.

In [ ]:
def create_csv_exchange_rate(response):

    # Parse the XML content returned by the API
    root = ET.fromstring(response.content)
    series = root.findall(".//{*}Series")

    # Store the extracted rows in an in-memory CSV buffer
    buffer = StringIO()

    for serie in series:
        currency = serie.attrib.get("CURRENCY")
        values = serie.findall(".//{*}Obs")
        for value in values:
            time_period = value.attrib.get("TIME_PERIOD")
            data_value = value.attrib.get("OBS_VALUE")
            buffer.write(f"{time_period},{currency},{data_value}\n")

    # Move the pointer to the beginning of the buffer
    buffer.seek(0)

    return buffer

#### Load Exchange Rate Data into PostgreSQL

This function inserts the exchange rate records stored in the in-memory buffer into the `exchange_rates` table.

Each row is processed individually and written to the database using an upsert strategy. If a record with the same `date` and `currency` already exists, the stored `rate` is updated with the new value. Once all rows have been processed, the transaction is committed.

In [ ]:
def load_exchange_rates(buffer, conn, cursor):

    # Ensure that the buffer is read from the beginning  
    buffer.seek(0)

    # Upsert exchange rate records into the target table
    query = """
        INSERT INTO exchange_rates (date, currency, rate)
        VALUES (%s, %s, %s)
        ON CONFLICT (date, currency)
        DO UPDATE SET rate = EXCLUDED.rate
    """

    # Read each line from the buffer and load it into the database
    for line in buffer:
        line = line.strip()
        if not line:
            continue

        date_value, currency, rate = line.split(",")

        cursor.execute(query, (date_value, currency, rate))

    conn.commit()

In [ ]:
update_new_data()

ParseError: no element found: line 1, column 0 (<string>)